In [1]:
import sys
import warnings

sys.path.append("..")
sys.dont_write_bytecode = True
warnings.filterwarnings("ignore")

In [2]:
import polars as pl
from utils.utils import (
    split_train_test,
    get_parent_run_id_from_experiment,
    compute_generalisation_error_from_run_id_and_experiment_id,
)
from utils.statics import lightgbm_model_name, FEATUERE_ENGINEERING_EXPERIMENT_ID
from utils.preprocessing_handler import PreprocessingHandler
from feature_engineering.RowWiseTransformations import RowWiseTransformations
from feature_engineering.OpenFETransformations import OpenFETransformations
from feature_engineering.ColumnTransformer import ColumnTransformer
from model.nested_cv_eval import ModelCVEvaluator

model_type = lightgbm_model_name

In [3]:
passes_df = pl.read_parquet("../data/02-analysis/passes.parquet")

In [4]:
train_df, test_df = split_train_test(passes_df=passes_df)

In [5]:
numerical_columns = [
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "length",
    "angle",
    "duration",
]

columns = train_df.columns
target_column = "outcome"
categorical_columns = [col for col in columns if col not in numerical_columns and col != target_column]

preprocessing_handler = PreprocessingHandler(df=train_df, categorical_columns=categorical_columns)
train_df_temp = preprocessing_handler.preprocess_categorical_columns()
train_df_temp = preprocessing_handler.preprocess_outcome_column()
y_train = train_df_temp.select("outcome").to_series()
X_train = preprocessing_handler.preprocess_outcome_column().drop(pl.col("outcome"))

In [6]:
categorical_columns = ["body_part", "height"]
column_transformer_config = {
            "cat_columns": categorical_columns
        }

In [7]:
row_wise_transformations = RowWiseTransformations()
column_wise_transformations = ColumnTransformer(**column_transformer_config)
open_fe_transformations = OpenFETransformations(n_features=5)

In [ ]:
run_name = f"Feature_engineering_11_OFE"
experiment_name = "Feature Engineering"
model_cv_evaluator = ModelCVEvaluator(
    model_type=model_type,
    row_wise_transformations=row_wise_transformations,
    column_wise_transformations=column_wise_transformations,
    open_fe_transformations=open_fe_transformations,
    n_inner_splits=3,
    n_outer_splits=3,
    n_trials=50,
    use_hyperparameter_tuning=False,
    use_feature_selection=False,
    use_feature_engineering=True,
    use_ofe=True,
    log_feature_importance=True,
    log_parameter_importance=True,
    run_name=run_name,
    experiment_name=experiment_name,
    categorical_columns=categorical_columns,
)

In [9]:
result_cv = model_cv_evaluator.get_generalisation_error(X_train=X_train, y_train=y_train)

2026-04-26 14:50:34,645 - INFO - Starting training with model lightgbm with the following configuration:
        - 3 inner splits
        - 2 outer splits
        - 50 trials
2026-04-26 14:50:35,565 - INFO - Fitting OpenFE on fold 1


The number of candidate features is 1775
Start stage I selection.


100%|██████████| 32/32 [00:18<00:00,  1.73it/s]


463 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 32/32 [00:52<00:00,  1.64s/it]


...

100%|██████████| 32/32 [00:23<00:00,  1.37it/s]


Finish data processing.


2026-04-26 14:54:36,025 - INFO - Feature: autoFE_f_0           | Formula: (duration+under_pressure)
2026-04-26 14:54:36,025 - INFO - Feature: autoFE_f_1           | Formula: (duration-angle_cos)
2026-04-26 14:54:36,025 - INFO - Feature: autoFE_f_2           | Formula: (length/direction_to_goal_cos)
2026-04-26 14:54:36,025 - INFO - Feature: ofe_col_1            | Formula: freq(duration)
2026-04-26 14:54:36,034 - INFO - Feature: ofe_col_2            | Formula: GroupByThenRank(duration,start_y)
2026-04-26 14:54:36,037 - INFO - Starting full hyperparameter tuning...
2026-04-26 14:54:36,037 - INFO - Fitting final model with best hyperparameters...


['body_part', 'height']


🏃 View run Outer_fold_2 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/3422a2685ece4602ae346999554a5b15
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708
🏃 View run Feature_engineering_11_OFE_lightgbm at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/ee29584d638c465ea84c3d6875fd590a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


In [10]:
parent_run_id = get_parent_run_id_from_experiment(result=result_cv, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID)
compute_generalisation_error_from_run_id_and_experiment_id(
    parent_run_id=parent_run_id, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID
)

Number of outer folds: 3
95% confidence interval for best estimate of generalisation: 0.2534289568910852 ± 0.00277424530519379
